In [2]:
import os
import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm
from skimage.feature import local_binary_pattern
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report
import matplotlib.pyplot as plt
import joblib


In [3]:
import os
import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm

IMAGE_FOLDER = r"C:\Users\Dell\Desktop\IIT\ds203\PROJECT\project_images\images_resized"   # <-- change this
EXCEL_PATH = r"C:\Users\Dell\Desktop\IIT\ds203\PROJECT\manual_classification.xlsx"  # <-- change this

df = pd.read_excel(EXCEL_PATH)
if "filename" in df.columns and "image_name" not in df.columns:
    df.rename(columns={"filename": "image_name"}, inplace=True)

PATCHES_FILE = "all_patches.npy"
LABELS_FILE = "all_labels.npy"

if os.path.exists(PATCHES_FILE) and os.path.exists(LABELS_FILE):
    print("✅ Found cached patch files. Loading...")
    all_patches = np.load(PATCHES_FILE, allow_pickle=True)
    all_labels = np.load(LABELS_FILE)
else:
    print("⚙️ Extracting 64 patches per image...")
    all_patches, all_labels = [], []

    for _, row in tqdm(df.iterrows(), total=len(df)):
        img_path = os.path.join(IMAGE_FOLDER, row["image_name"])
        image = cv2.imread(img_path)
        if image is None:
            continue

        # Ensure fixed size (800x600)
        image = cv2.resize(image, (800, 600))

        h, w, _ = image.shape
        ph, pw = h // 8, w // 8

        for block_id in range(64):
            i, j = divmod(block_id, 8)
            y1, y2 = i * ph, (i + 1) * ph
            x1, x2 = j * pw, (j + 1) * pw
            patch = image[y1:y2, x1:x2]

            # Ensure consistent patch size (should be 75x100)
            patch = cv2.resize(patch, (100, 75))

            all_patches.append(patch)
            all_labels.append(int(row[f"block_{block_id}"]))

    all_patches = np.array(all_patches, dtype=object)
    all_labels = np.array(all_labels)

    np.save(PATCHES_FILE, all_patches, allow_pickle=True)
    np.save(LABELS_FILE, all_labels)
    print(f"✅ Saved all patches to {PATCHES_FILE} and labels to {LABELS_FILE}")

print("Patches:", len(all_patches))
print("Labels:", len(all_labels))
print("Example patch shape:", all_patches[0].shape)


✅ Found cached patch files. Loading...
Patches: 29760
Labels: 29760
Example patch shape: (75, 100, 3)


In [5]:
def extract_hsv_features(patch):
    hsv = cv2.cvtColor(patch, cv2.COLOR_BGR2HSV)
    hist = cv2.calcHist([hsv], [0, 1, 2], None, [8, 8, 8],
                        [0, 180, 0, 256, 0, 256])
    return cv2.normalize(hist, hist).flatten()

def extract_lbp_features(patch):
    gray = cv2.cvtColor(patch, cv2.COLOR_BGR2GRAY)
    lbp = local_binary_pattern(gray, 8, 1, method='uniform')
    hist, _ = np.histogram(lbp.ravel(), bins=np.arange(0, 11), range=(0, 10))
    return hist / np.sum(hist)

def extract_laplacian_features(patch):
    gray = cv2.cvtColor(patch, cv2.COLOR_BGR2GRAY)
    lap = cv2.Laplacian(gray, cv2.CV_64F)
    hist, _ = np.histogram(lap.ravel(), bins=32, range=(-255, 255))
    return hist / np.sum(hist)


In [10]:
X_CACHE = "X_hsv.npy"
Y_CACHE = "y_labels.npy"

if os.path.exists(X_CACHE) and os.path.exists(Y_CACHE):
    print("✅ Found cached HSV features. Skipping extraction.")
    X = np.load(X_CACHE)
    y = np.load(Y_CACHE)
else:
    print("⚙️ Extracting HSV features...")

    all_patches = np.load("all_patches.npy", allow_pickle=True)
    all_labels = np.load("all_labels.npy")

    # ✅ Fix for dtype=object issue
    valid_patches = []
    for p in all_patches:
        if isinstance(p, np.ndarray) and p.ndim == 3:
            valid_patches.append(p.astype(np.uint8))
    all_patches = np.array(valid_patches)
    print(f"✅ Loaded {len(all_patches)} valid patches.")

    X, y = [], all_labels[:len(all_patches)]  # sync lengths
    for patch in tqdm(all_patches):
        feats = extract_hsv_features(patch)
        X.append(feats)

    X = np.array(X)
    y = np.array(y)

    np.save(X_CACHE, X)
    np.save(Y_CACHE, y)
    print("✅ HSV features saved!")

print("Feature matrix shape:", X.shape)


⚙️ Extracting HSV features...
✅ Loaded 29760 valid patches.


100%|██████████| 29760/29760 [00:03<00:00, 8286.56it/s]


✅ HSV features saved!
Feature matrix shape: (29760, 512)


In [11]:
import numpy as np
from tqdm import tqdm
import cv2
from skimage.feature import local_binary_pattern

# --- Functions for feature extraction ---
def extract_lbp_features(patch):
    gray = cv2.cvtColor(patch, cv2.COLOR_BGR2GRAY)
    lbp = local_binary_pattern(gray, 8, 1, method='uniform')
    hist, _ = np.histogram(lbp.ravel(), bins=np.arange(0, 11), range=(0, 10))
    return (hist / np.sum(hist)).astype("float32")

def extract_laplacian_features(patch):
    gray = cv2.cvtColor(patch, cv2.COLOR_BGR2GRAY)
    lap = cv2.Laplacian(gray, cv2.CV_64F)
    stats = [lap.mean(), lap.var()]
    return np.array(stats, dtype="float32")

# --- Load saved patches and labels ---
all_patches = np.load("all_patches.npy", allow_pickle=True)
all_labels = np.load("all_labels.npy", allow_pickle=True)

print(f"Loaded {len(all_patches)} patches.")

X_extra = []
valid_labels = []

for i, patch in enumerate(tqdm(all_patches, desc="⚙️ Computing LBP + Laplacian")):
    try:
        patch = np.array(patch, dtype=np.uint8)  # ensure valid array
        if patch.ndim != 3:
            raise ValueError("Patch not 3D")

        lbp_feats = extract_lbp_features(patch)
        lap_feats = extract_laplacian_features(patch)
        feats = np.concatenate([lbp_feats, lap_feats])
        X_extra.append(feats)
        valid_labels.append(all_labels[i])
    except Exception as e:
        print(f"⚠️ Skipping patch {i}: {e}")
        continue

X_extra = np.array(X_extra)
y_valid = np.array(valid_labels)

# Combine with saved HSV features
X_hsv = np.load("X_hsv.npy")
min_len = min(len(X_hsv), len(X_extra))  # sync in case of skipped patches
X_full = np.hstack([X_hsv[:min_len], X_extra[:min_len]])
y_valid = y_valid[:min_len]

# Save combined dataset
np.save("X_full.npy", X_full)
np.save("y_labels.npy", y_valid)
print(f"✅ Saved combined features: X_full.npy ({X_full.shape}), y_labels.npy")


Loaded 29760 patches.


⚙️ Computing LBP + Laplacian: 100%|██████████| 29760/29760 [02:39<00:00, 186.58it/s]


✅ Saved combined features: X_full.npy ((29760, 524)), y_labels.npy


In [10]:
# --- ⚙️ Imports ---
import numpy as np
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, classification_report
)
import joblib, warnings
warnings.filterwarnings("ignore")

# --- 📂 Load feature sets ---
# X = np.load("X_hsv.npy")      # Only HSV features
X = np.load("X_full.npy")       # HSV + LBP + Laplacian
y = np.load("y_labels.npy")

print(f"✅ Dataset loaded: X={X.shape}, y={y.shape}")

# --- 🧩 Train/Test split ---
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# --- ⚖️ Scale features ---
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# --- ⚡ Speed optimization: train on subset if dataset is huge ---
MAX_TRAIN_SAMPLES = 30000  # adjust: keeps good accuracy & fast training
if len(X_train_scaled) > MAX_TRAIN_SAMPLES:
    import numpy as np
    idx = np.random.choice(len(X_train_scaled), MAX_TRAIN_SAMPLES, replace=False)
    X_train_sub = X_train_scaled[idx]
    y_train_sub = y_train[idx]
    print(f"🧠 Using subset of {len(X_train_sub)} samples for faster training")
else:
    X_train_sub, y_train_sub = X_train_scaled, y_train

# --- 🧮 Train moderately tuned RBF SVM ---
svm = SVC(
    kernel="rbf",
    probability=True,
    random_state=42,
    C=2.0,              # slightly higher C = better fit
    gamma="scale",
    cache_size=1000,    # speed up kernel caching
    max_iter=1000       # stops if taking too long
)
svm.fit(X_train_sub, y_train_sub)

# --- 🔍 Evaluate ---
y_pred = svm.predict(X_test_scaled)
y_prob = svm.predict_proba(X_test_scaled)[:, 1]

print("\n📊 Evaluation (Test Set):")
print(f"Accuracy : {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred):.4f}")
print(f"Recall   : {recall_score(y_test, y_pred):.4f}")
print(f"F1 Score : {f1_score(y_test, y_pred):.4f}")
print(f"AUC      : {roc_auc_score(y_test, y_prob):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# --- 💾 Save model ---
joblib.dump(svm, "svm_rbf_fast.pkl")
joblib.dump(scaler, "scaler.pkl")
print("✅ Model and scaler saved as 'svm_rbf_fast.pkl' and 'scaler.pkl'")


✅ Dataset loaded: X=(29760, 524), y=(29760,)

📊 Evaluation (Test Set):
Accuracy : 0.3505
Precision: 0.2206
Recall   : 0.7019
F1 Score : 0.3357
AUC      : 0.4911

Classification Report:
              precision    recall  f1-score   support

           0       0.73      0.24      0.36      4560
           1       0.22      0.70      0.34      1392

    accuracy                           0.35      5952
   macro avg       0.47      0.47      0.35      5952
weighted avg       0.61      0.35      0.36      5952

✅ Model and scaler saved as 'svm_rbf_fast.pkl' and 'scaler.pkl'


In [4]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os

def extract_hsv_features(patch):
    hsv = cv2.cvtColor(patch, cv2.COLOR_BGR2HSV)
    hist = cv2.calcHist([hsv], [0, 1, 2], None, [8, 8, 8],
                        [0, 180, 0, 256, 0, 256])
    return cv2.normalize(hist, hist).flatten()

def extract_lbp_features(patch):
    gray = cv2.cvtColor(patch, cv2.COLOR_BGR2GRAY)
    lbp = local_binary_pattern(gray, P=8, R=1, method='uniform')
    hist, _ = np.histogram(lbp.ravel(), bins=np.arange(0, 11), range=(0, 10))
    hist = hist.astype("float")
    hist /= (hist.sum() + 1e-6)
    return hist  # 10 features

def extract_laplacian_features(patch):
    gray = cv2.cvtColor(patch, cv2.COLOR_BGR2GRAY)
    lap = cv2.Laplacian(gray, cv2.CV_64F)
    return np.array([lap.mean(), lap.var(), lap.std()])  # 3 features


def visualize_predictions(img_path, model, scaler):
    image = cv2.imread(img_path)
    if image is None:
        return None

    h, w, _ = image.shape
    ph, pw = h // 8, w // 8
    overlay = image.copy()

    n_expected = scaler.n_features_in_

    for i in range(8):
        for j in range(8):
            y1, y2 = i * ph, (i + 1) * ph
            x1, x2 = j * pw, (j + 1) * pw
            patch = image[y1:y2, x1:x2]

            # Always compute HSV
            hsv_feats = extract_hsv_features(patch)
            feats = hsv_feats

            # Add others only if model expects them
            if n_expected > len(hsv_feats):
                lbp_feats = extract_lbp_features(patch)
                lap_feats = extract_laplacian_features(patch)
                feats = np.concatenate([hsv_feats, lbp_feats, lap_feats])

            # --- Auto-adjust to expected size ---
            if len(feats) > n_expected:
                feats = feats[:n_expected]
            elif len(feats) < n_expected:
                feats = np.pad(feats, (0, n_expected - len(feats)), mode="constant")

            feats_scaled = scaler.transform([feats])
            pred = model.predict(feats_scaled)[0]
            color = (0, 0, 255) if pred == 1 else (255, 255, 255)
            cv2.rectangle(overlay, (x1, y1), (x2, y2), color, 2)

    return cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB)



# --- Show 20 random images with grid predictions ---
sample_images = df["image_name"].sample(20, random_state=42).tolist()

plt.figure(figsize=(15, 15))
for idx, filename in enumerate(sample_images):
    img_path = os.path.join(IMAGE_FOLDER, filename)
    vis = visualize_predictions(img_path, svm, scaler)
    if vis is not None:
        plt.subplot(5, 4, idx + 1)
        plt.imshow(vis)
        plt.axis("off")
        plt.title(filename)
plt.tight_layout()
plt.show()


NameError: name 'svm' is not defined

<Figure size 1500x1500 with 0 Axes>